# Text-to-SQL em português: fine-tuning de LLM com QLoRA (GPU)

Notebook para rodar no **Google Colab** (Ambiente de execução → Alterar tipo → **T4 GPU**).
Fluxo: verificar GPU → dataset → baseline → 4 treinos com parâmetros diferentes → avaliação → frontend.

## 1. Verificar a GPU (print exigido no CP)

In [ ]:
!nvidia-smi
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

## 2. Obter o código
Troque `SEU_USUARIO` pelo seu usuário do GitHub (o repositório precisa estar publicado).

In [ ]:
GITHUB_REPO = "https://github.com/palomari/text2sql-llm.git"
!git clone {GITHUB_REPO}
%cd text2sql-llm

In [ ]:
!pip install -q -r requirements.txt

## 3. Criar banco e dataset

In [ ]:
!python src/make_dataset.py
!head -n 3 data/train.jsonl

## 4. Baseline: modelo SEM fine-tuning

In [ ]:
!python src/evaluate_model.py --name baseline

## 5. Experimentos com parâmetros diferentes
Cada execução muda **um** parâmetro em relação à `run1`. Os resultados vão para `results/experimentos.csv`.

In [ ]:
!python src/train.py --name run1 --r 8  --lr 2e-4 --epochs 3

In [ ]:
!python src/train.py --name run2 --r 16 --lr 2e-4 --epochs 3

In [ ]:
!python src/train.py --name run3 --r 8  --lr 1e-4 --epochs 3

In [ ]:
!python src/train.py --name run4 --r 8  --lr 2e-4 --epochs 5

## 6. Avaliar cada execução no conjunto de teste

In [ ]:
for r in ["run1", "run2", "run3", "run4"]:
    !python src/evaluate_model.py --name {r} --adapter outputs/{r}/final

## 7. Parâmetros de inferência (temperature) na melhor execução
Troque `run1` pela melhor execução.

In [ ]:
BEST = "run1"
for t in [0.0, 0.3, 0.7, 1.0]:
    !python src/evaluate_model.py --name {BEST} --adapter outputs/{BEST}/final --temperature {t}

## 8. Tabelas e gráfico

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
exp = pd.read_csv("results/experimentos.csv")
av = pd.read_csv("results/avaliacao.csv")
display(exp)
display(av)

d = av[av.temperature == 0.0]
plt.figure(figsize=(7, 4))
plt.bar(d.run, d.exec_accuracy)
plt.ylabel("Accuracy por execução"); plt.title("Baseline vs. fine-tuning (teste)")
plt.ylim(0, 1); plt.tight_layout()
plt.savefig("results/comparacao.png", dpi=150)
plt.show()

## 9. Exemplos qualitativos (baseline x fine-tuned)

In [ ]:
import json
base = json.load(open("results/pred_baseline_T0.0.json", encoding="utf-8"))
ft = json.load(open(f"results/pred_{BEST}_T0.0.json", encoding="utf-8"))
for b, f in list(zip(base, ft))[:8]:
    print("Pergunta :", b["question"])
    print("Esperada :", b["gold_sql"])
    print("Baseline :", b["pred_sql"], "->", "OK" if b["correto"] else "ERRO")
    print("Fine-tune:", f["pred_sql"], "->", "OK" if f["correto"] else "ERRO")
    print("-" * 80)

## 10. Frontend (Gradio)
Gera um link público temporário. Tire o print do frontend funcionando.

In [ ]:
!python src/app.py --adapter outputs/{BEST}/final --share

## 11. Salvar resultados no Google Drive (a sessão do Colab é temporária)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/text2sql-llm
!cp -r results outputs /content/drive/MyDrive/text2sql-llm/